In [1]:
import os
import sys

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Crime Data Analysis and Prediction")
    .master("local[*]")
    .config("spark.python.worker.reuse", "true")
    .getOrCreate()
)

print("Python:", sys.version)
print("Spark:", spark.version)

Python: 3.11.15 | packaged by Anaconda, Inc. | (main, Jun 11 2026, 15:12:53) [MSC v.1942 64 bit (AMD64)]
Spark: 4.2.0


In [4]:
data = [
    ("THEFT", "District 1"),
    ("BATTERY", "District 2"),
    ("THEFT", "District 1"),
    ("ROBBERY", "District 7")
]

columns = ["Crime_Type", "District"]

test_df = spark.createDataFrame(data, columns)

test_df.show()

+----------+----------+
|Crime_Type|  District|
+----------+----------+
|     THEFT|District 1|
|   BATTERY|District 2|
|     THEFT|District 1|
|   ROBBERY|District 7|
+----------+----------+



In [5]:
test_df.groupBy("Crime_Type").count().show()

+----------+-----+
|Crime_Type|count|
+----------+-----+
|     THEFT|    2|
|   BATTERY|    1|
|   ROBBERY|    1|
+----------+-----+



# 1. Dataset Loading
## Loading the Chicago Crime Dataset using Apache Spark

In [2]:
crime_df = spark.read.csv(
    "data/Crimes_-_2001_to_Present.csv",
    header=True,
    inferSchema=True
)

print("Dataset loaded successfully!")

Dataset loaded successfully!


In [7]:
crime_df.printSchema()

root
 |-- ID: integer (nullable = true)
 |-- Case Number: string (nullable = true)
 |-- Date: string (nullable = true)
 |-- Block: string (nullable = true)
 |-- IUCR: string (nullable = true)
 |-- Primary Type: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Location Description: string (nullable = true)
 |-- Arrest: boolean (nullable = true)
 |-- Domestic: boolean (nullable = true)
 |-- Beat: integer (nullable = true)
 |-- District: integer (nullable = true)
 |-- Ward: integer (nullable = true)
 |-- Community Area: integer (nullable = true)
 |-- FBI Code: string (nullable = true)
 |-- X Coordinate: integer (nullable = true)
 |-- Y Coordinate: integer (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Updated On: string (nullable = true)
 |-- Latitude: double (nullable = true)
 |-- Longitude: double (nullable = true)
 |-- Location: string (nullable = true)



In [8]:
total_records = crime_df.count()

print("Total crime records:", total_records)

Total crime records: 7784664


In [9]:
crime_df.show(5, truncate=False)

+--------+-----------+----------------------+---------------------+----+------------+-----------------------+--------------------+------+--------+----+--------+----+--------------+--------+------------+------------+----+----------------------+------------+-------------+-----------------------------+
|ID      |Case Number|Date                  |Block                |IUCR|Primary Type|Description            |Location Description|Arrest|Domestic|Beat|District|Ward|Community Area|FBI Code|X Coordinate|Y Coordinate|Year|Updated On            |Latitude    |Longitude    |Location                     |
+--------+-----------+----------------------+---------------------+----+------------+-----------------------+--------------------+------+--------+----+--------+----+--------------+--------+------------+------------+----+----------------------+------------+-------------+-----------------------------+
|10224738|HY411648   |09/05/2015 01:30:00 PM|043XX S WOOD ST      |0486|BATTERY     |DOMESTIC BAT

# 2. Data Understanding and Quality Analysis
## 2.1 Dataset Dimensions

In [10]:
# Number of rows
num_rows = crime_df.count()

# Number of columns
num_columns = len(crime_df.columns)

print("Number of rows:", num_rows)
print("Number of columns:", num_columns)

Number of rows: 7784664
Number of columns: 22


## 2.2 Dataset Columns


In [11]:
for i, column in enumerate(crime_df.columns, start=1):
    print(f"{i}. {column}")

1. ID
2. Case Number
3. Date
4. Block
5. IUCR
6. Primary Type
7. Description
8. Location Description
9. Arrest
10. Domestic
11. Beat
12. District
13. Ward
14. Community Area
15. FBI Code
16. X Coordinate
17. Y Coordinate
18. Year
19. Updated On
20. Latitude
21. Longitude
22. Location


## 2.3 Dataset Schema

In [12]:
crime_df.printSchema()

root
 |-- ID: integer (nullable = true)
 |-- Case Number: string (nullable = true)
 |-- Date: string (nullable = true)
 |-- Block: string (nullable = true)
 |-- IUCR: string (nullable = true)
 |-- Primary Type: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Location Description: string (nullable = true)
 |-- Arrest: boolean (nullable = true)
 |-- Domestic: boolean (nullable = true)
 |-- Beat: integer (nullable = true)
 |-- District: integer (nullable = true)
 |-- Ward: integer (nullable = true)
 |-- Community Area: integer (nullable = true)
 |-- FBI Code: string (nullable = true)
 |-- X Coordinate: integer (nullable = true)
 |-- Y Coordinate: integer (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Updated On: string (nullable = true)
 |-- Latitude: double (nullable = true)
 |-- Longitude: double (nullable = true)
 |-- Location: string (nullable = true)



## 2.4 Sample Records

In [13]:
crime_df.show(10, truncate=False)

+--------+-----------+----------------------+-----------------------+----+------------------+-----------------------------------+--------------------+------+--------+----+--------+----+--------------+--------+------------+------------+----+----------------------+------------+-------------+-----------------------------+
|ID      |Case Number|Date                  |Block                  |IUCR|Primary Type      |Description                        |Location Description|Arrest|Domestic|Beat|District|Ward|Community Area|FBI Code|X Coordinate|Y Coordinate|Year|Updated On            |Latitude    |Longitude    |Location                     |
+--------+-----------+----------------------+-----------------------+----+------------------+-----------------------------------+--------------------+------+--------+----+--------+----+--------------+--------+------------+------------+----+----------------------+------------+-------------+-----------------------------+
|10224738|HY411648   |09/05/2015 01:3

## 2.5 Missing-Value Analysis

In [14]:
from pyspark.sql.functions import col, sum, when, count

missing_values = crime_df.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in crime_df.columns
])

missing_values.show(truncate=False)

+---+-----------+----+-----+----+------------+-----------+--------------------+------+--------+----+--------+------+--------------+--------+------------+------------+----+----------+--------+---------+--------+
|ID |Case Number|Date|Block|IUCR|Primary Type|Description|Location Description|Arrest|Domestic|Beat|District|Ward  |Community Area|FBI Code|X Coordinate|Y Coordinate|Year|Updated On|Latitude|Longitude|Location|
+---+-----------+----+-----+----+------------+-----------+--------------------+------+--------+----+--------+------+--------------+--------+------------+------------+----+----------+--------+---------+--------+
|0  |4          |0   |0    |0   |0           |0          |10381               |0     |0       |0   |47      |614848|613476        |0       |86848       |86848       |0   |0         |86848   |86848    |86848   |
+---+-----------+----+-----+----+------------+-----------+--------------------+------+--------+----+--------+------+--------------+--------+------------+---

## 2.6 Missing-Value Percentage

In [15]:
from pyspark.sql.functions import col, sum, when, round

total_records = crime_df.count()

missing_percentage = crime_df.select([
    round(
        sum(when(col(c).isNull(), 1).otherwise(0)) / total_records * 100,
        2
    ).alias(c)
    for c in crime_df.columns
])

missing_percentage.show(truncate=False)

+---+-----------+----+-----+----+------------+-----------+--------------------+------+--------+----+--------+----+--------------+--------+------------+------------+----+----------+--------+---------+--------+
|ID |Case Number|Date|Block|IUCR|Primary Type|Description|Location Description|Arrest|Domestic|Beat|District|Ward|Community Area|FBI Code|X Coordinate|Y Coordinate|Year|Updated On|Latitude|Longitude|Location|
+---+-----------+----+-----+----+------------+-----------+--------------------+------+--------+----+--------+----+--------------+--------+------------+------------+----+----------+--------+---------+--------+
|0.0|0.0        |0.0 |0.0  |0.0 |0.0         |0.0        |0.13                |0.0   |0.0     |0.0 |0.0     |7.9 |7.88          |0.0     |1.12        |1.12        |0.0 |0.0       |1.12    |1.12     |1.12    |
+---+-----------+----+-----+----+------------+-----------+--------------------+------+--------+----+--------+----+--------------+--------+------------+------------+

In [18]:
from pyspark.sql.functions import col, sum, when
import builtins

total_records = crime_df.count()

missing_data = crime_df.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in crime_df.columns
])

missing_rows = missing_data.collect()[0]

missing_summary = []

for column, count_missing in zip(crime_df.columns, missing_rows):
    if count_missing > 0:
        percentage = (count_missing / total_records) * 100
        
        missing_summary.append(
            (column, int(count_missing), builtins.round(percentage, 2))
        )

missing_df = spark.createDataFrame(
    missing_summary,
    ["Column", "Missing_Count", "Missing_Percentage"]
)

missing_df.orderBy(
    col("Missing_Percentage").desc()
).show(truncate=False)

+--------------------+-------------+------------------+
|Column              |Missing_Count|Missing_Percentage|
+--------------------+-------------+------------------+
|Ward                |614848       |7.9               |
|Community Area      |613476       |7.88              |
|X Coordinate        |86848        |1.12              |
|Location            |86848        |1.12              |
|Latitude            |86848        |1.12              |
|Longitude           |86848        |1.12              |
|Y Coordinate        |86848        |1.12              |
|Location Description|10381        |0.13              |
|District            |47           |0.0               |
|Case Number         |4            |0.0               |
+--------------------+-------------+------------------+



## 2.7 Approximate Record Analysis

In [4]:
from pyspark.sql.functions import approx_count_distinct

total_records = crime_df.count()

approx_unique_ids = crime_df.select(
    approx_count_distinct("ID", rsd=0.01).alias("approx_unique_ids")
).collect()[0]["approx_unique_ids"]

print("Total records:", total_records)
print("Approximate unique IDs:", approx_unique_ids)

Total records: 7784664
Approximate unique IDs: 7818643


## 2.8 Categorical Value Analysis
### Crime Types

In [6]:
from pyspark.sql.functions import col
crime_df.groupBy("Primary Type") \
    .count() \
    .orderBy(col("count").desc()) \
    .show(truncate=False)

+--------------------------------+-------+
|Primary Type                    |count  |
+--------------------------------+-------+
|THEFT                           |1642148|
|BATTERY                         |1422913|
|CRIMINAL DAMAGE                 |887266 |
|NARCOTICS                       |747633 |
|ASSAULT                         |507296 |
|OTHER OFFENSE                   |483642 |
|BURGLARY                        |424397 |
|MOTOR VEHICLE THEFT             |375495 |
|DECEPTIVE PRACTICE              |344940 |
|ROBBERY                         |292334 |
|CRIMINAL TRESPASS               |214316 |
|WEAPONS VIOLATION               |106418 |
|PROSTITUTION                    |69840  |
|OFFENSE INVOLVING CHILDREN      |55719  |
|PUBLIC PEACE VIOLATION          |52325  |
|SEX OFFENSE                     |30683  |
|CRIM SEXUAL ASSAULT             |27631  |
|INTERFERENCE WITH PUBLIC OFFICER|18392  |
|LIQUOR LAW VIOLATION            |14901  |
|GAMBLING                        |14618  |
+----------

### Arrest Distribution

In [7]:
crime_df.groupBy("Arrest") \
    .count() \
    .orderBy(col("count").desc()) \
    .show()

+------+-------+
|Arrest|  count|
+------+-------+
| false|5749900|
|  true|2034764|
+------+-------+



In [8]:
crime_df.groupBy("Domestic") \
    .count() \
    .orderBy(col("count").desc()) \
    .show()

+--------+-------+
|Domestic|  count|
+--------+-------+
|   false|6708370|
|    true|1076294|
+--------+-------+



In [9]:
crime_df.select(
    "Latitude",
    "Longitude",
    "X Coordinate",
    "Y Coordinate"
).summary().show()

+-------+------------------+-------------------+------------------+------------------+
|summary|          Latitude|          Longitude|      X Coordinate|      Y Coordinate|
+-------+------------------+-------------------+------------------+------------------+
|  count|           7697816|            7697816|           7697816|           7697816|
|   mean|41.842183638280936| -87.67149303901566|1164601.2705625854|1885782.8573099175|
| stddev|0.0887959834316862|0.06108257002119852|16846.578922131077|32275.312527212514|
|    min|      36.619446395|      -91.686565684|                 0|                 0|
|    25%|      41.768705597|      -87.713673374|           1152976|           1859069|
|    50%|      41.855906242|      -87.665844597|           1166110|           1890728|
|    75%|      41.906765676|      -87.628194981|           1176371|           1909272|
|    max|      42.022910333|      -87.524529378|           1205119|           1951622|
+-------+------------------+---------------

# 3. Data Cleaning

## 3.1 Handling Missing Categorical Values

In [14]:
from pyspark.sql.functions import col

crime_clean_df = crime_df.fillna({
    "Location Description": "Unknown"
})

print("Categorical missing values handled.")

Categorical missing values handled.


In [15]:
crime_clean_df.select(
    "Location Description",
    "District",
    "Ward",
    "Community Area"
).show(10, truncate=False)

+--------------------+--------+----+--------------+
|Location Description|District|Ward|Community Area|
+--------------------+--------+----+--------------+
|RESIDENCE           |9       |12  |61            |
|CTA BUS             |15      |29  |25            |
|RESIDENCE           |6       |8   |44            |
|SIDEWALK            |14      |35  |21            |
|APARTMENT           |15      |28  |25            |
|RESIDENCE           |6       |21  |71            |
|RESIDENCE-GARAGE    |14      |32  |24            |
|GROCERY FOOD STORE  |10      |25  |31            |
|STREET              |12      |27  |27            |
|Unknown             |8       |15  |63            |
+--------------------+--------+----+--------------+
only showing top 10 rows


In [16]:
from pyspark.sql.functions import sum, when

crime_clean_df.select([
    sum(
        when(col(c).isNull(), 1).otherwise(0)
    ).alias(c)
    for c in [
        "Location Description",
        "District",
        "Ward",
        "Community Area"
    ]
]).show()

+--------------------+--------+------+--------------+
|Location Description|District|  Ward|Community Area|
+--------------------+--------+------+--------------+
|                   0|      47|614848|        613476|
+--------------------+--------+------+--------------+



In [17]:
original_count = crime_df.count()
cleaned_count = crime_clean_df.count()

print("Original records:", original_count)
print("Cleaned records:", cleaned_count)
print("Records removed:", original_count - cleaned_count)

Original records: 7784664
Cleaned records: 7784664
Records removed: 0
